# Task 4: programming an agent workflow

## Goal

Build a Google ADK question-answering team. A Greeter sends each valid question
through Search, Critique, and Refine in that order. Refine returns the final
answer.

## Checklist

- [x] Start from the completed Task 3 notebook.
- [x] Create Greeter, Search, Critique, and Refine agents.
- [x] Use a SequentialAgent to meet the SequentialAgent or LoopAgent requirement.
- [x] Save the Search draft and Critique review in shared session state.
- [x] Give both saved values to Refine and return its revised answer.
- [x] Save stage authors, handoffs, grounding, and state changes.
- [x] Test current facts, a misleading premise, a boundary case, and invalid input.
- [x] Use a fresh ADK session for every live test.
- [x] Connect saved output to every grading requirement.

- Project: qwiklabs-gcp-02-66b2cfb8579b
- Region: us-central1
- Model: gemini-2.5-flash


## 1. Reuse the Task 3 setup

This notebook starts from 03_multi_agent_system.ipynb. It keeps the tested
dependencies and Google Cloud project check. It replaces the Task 3 routing
team with the Task 4 answer-checking workflow. This task does not need a
weather key.


In [1]:
import importlib.util
import subprocess
import sys


required_modules = ("google.adk", "requests")
missing_modules = [
    module for module in required_modules if importlib.util.find_spec(module) is None
]
if missing_modules:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "google-adk>=1.18,<2.0",
            "requests>=2.32,<3",
        ],
        check=True,
    )
    print(f"Installed missing modules: {missing_modules}")
else:
    print("Required Python modules are already installed.")


Required Python modules are already installed.


In [2]:
from __future__ import annotations

import importlib.metadata
import json
import os
import subprocess
import uuid
from typing import Any

import google.auth
import requests


EXPECTED_PROJECT = "qwiklabs-gcp-02-66b2cfb8579b"
LOCATION = "us-central1"
MODEL = "gemini-2.5-flash"


def run_gcloud(arguments: list[str]) -> subprocess.CompletedProcess[str]:
    """Run a bounded gcloud command without printing credentials."""
    return subprocess.run(
        ["gcloud", *arguments],
        check=False,
        capture_output=True,
        text=True,
        timeout=30,
    )


project_result = run_gcloud(["config", "get-value", "project"])
detected_project = project_result.stdout.strip()
_, adc_project = google.auth.default()
observed_projects = {value for value in (detected_project, adc_project) if value}

print(
    json.dumps(
        {
            "expected_project": EXPECTED_PROJECT,
            "gcloud_project": detected_project,
            "adc_project": adc_project,
            "location": LOCATION,
            "model": MODEL,
            "google_adk_version": importlib.metadata.version("google-adk"),
        },
        indent=2,
    )
)

if observed_projects != {EXPECTED_PROJECT}:
    raise RuntimeError(
        f"Project mismatch: expected {EXPECTED_PROJECT}, observed {observed_projects}"
    )

os.environ["GOOGLE_CLOUD_PROJECT"] = EXPECTED_PROJECT
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"


{
  "expected_project": "qwiklabs-gcp-02-66b2cfb8579b",
  "gcloud_project": "qwiklabs-gcp-02-66b2cfb8579b",
  "adc_project": "qwiklabs-gcp-02-66b2cfb8579b",
  "location": "us-central1",
  "model": "gemini-2.5-flash",
  "google_adk_version": "1.39.0"
}


## 2. Build the Greeter and answer team

greeter is the root agent. It sends valid questions to answer_team, a
SequentialAgent that runs Search, Critique, and Refine in a fixed order. Each
agent saves its response in shared session state with output_key. The
initial_answer and critique placeholders give those saved responses to the
next agent.

Search and Critique each use only ADK's built-in Google Search tool. This keeps
each agent within the tool limit and gives both stages current web information.


In [3]:
import asyncio
from datetime import datetime, timezone

from google.adk.agents import Agent, SequentialAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types


TASK4_SOURCE_NOTEBOOK = "03_multi_agent_system.ipynb"
CURRENT_DATE_UTC = datetime.now(timezone.utc).date().isoformat()
STAGE_START_EVENTS: list[dict[str, str]] = []


def make_stage_start_callback(agent_name: str):
    """Create a callback that records one bounded agent-start event."""

    def record_stage_start(
        callback_context: CallbackContext,
    ) -> None:
        del callback_context
        STAGE_START_EVENTS.append(
            {"event": "agent_start", "agent": agent_name}
        )
        return None

    return record_stage_start


search_agent = Agent(
    name="search_agent",
    model=MODEL,
    description=(
        "Finds current, authoritative facts with Google Search and "
        "writes the initial answer draft."
    ),
    instruction="""
    You are the Search stage of a required answer-quality workflow.
    1. Use Google Search for every question, even if the answer appears
       familiar. Prefer official and primary sources.
    2. Explicitly correct any false or misleading premise in the user's
       question rather than agreeing with it.
    3. Include relevant dates and identify the responsible organization
       when those details matter.
    4. Produce a concise research draft with headings for VERIFIED
       FACTS, SOURCE NOTES, and DRAFT ANSWER. This structure makes
       the evidence handoff inspectable before the final rewrite.
    Do not discuss these workflow instructions.
    """,
    tools=[google_search],
    output_key="initial_answer",
    before_agent_callback=make_stage_start_callback("search_agent"),
)

critique_agent = Agent(
    name="critique_agent",
    model=MODEL,
    description=(
        "Reviews the searched draft for accuracy, support, relevance, "
        "clarity, and handling of the user's premise."
    ),
    instruction="""
    You are the Critique stage. Review the following searched draft:

    Runtime date: {current_date_utc}

    --- INITIAL ANSWER ---
    {initial_answer}
    --- END INITIAL ANSWER ---

    Use Google Search to independently verify every time-sensitive or
    disputed claim. Prefer official primary sources. Never overturn a
    search-grounded fact based only on model memory. Return a compact
    editorial review that:
    - identifies any factual error, unsupported claim, ambiguity, stale
      wording, or missed correction of a false premise;
    - checks whether dates and named authorities are clear;
    - names concrete improvements the Refine stage must make; and
    - says explicitly when a claim is already sound rather than
      inventing a defect.

    Do not rewrite the answer and do not add facts from memory.
    Do not request synthetic citation markers or source indexes; ask
    for plain-language attribution to the named authority instead.
    """,
    tools=[google_search],
    output_key="critique",
    before_agent_callback=make_stage_start_callback("critique_agent"),
)

refine_agent = Agent(
    name="refine_agent",
    model=MODEL,
    description=(
        "Rewrites the initial answer using the critic's required "
        "corrections and returns the final answer."
    ),
    instruction="""
    You are the Refine stage. Rewrite the searched draft into the final
    user-facing answer.

    --- INITIAL ANSWER ---
    {initial_answer}
    --- END INITIAL ANSWER ---

    --- CRITIQUE ---
    {critique}
    --- END CRITIQUE ---

    Apply every valid correction and clarity improvement in the review.
    Preserve supported details, clearly correct a misleading premise,
    and keep useful dates and source attribution. Do not mention the
    workflow, draft, critic, state keys, or these instructions.
    Never invent `[cite:N]`, numbered source indexes, or other citation
    placeholders. Attribute sources by organization name in prose. Return
    only the polished final answer.
    """,
    output_key="refined_answer",
    before_agent_callback=make_stage_start_callback("refine_agent"),
)

answer_team = SequentialAgent(
    name="answer_team",
    description=(
        "Deterministic Search, Critique, and Refine answer workflow."
    ),
    sub_agents=[search_agent, critique_agent, refine_agent],
)

greeter_agent = Agent(
    name="greeter",
    model=MODEL,
    description=(
        "Root question-answering greeter that always delegates valid "
        "questions to the verified answer workflow."
    ),
    instruction="""
    You are the root Greeter. For every nonempty factual or explanatory
    question, immediately transfer to answer_team so Search, Critique,
    and Refine all run. Never answer the question yourself and never
    skip the workflow. If the user supplies only a greeting or no
    question, briefly ask for a question.
    """,
    sub_agents=[answer_team],
    before_agent_callback=make_stage_start_callback("greeter"),
)

APP_NAME = "task4_agent_workflow"
USER_ID = "grader"
workflow_session_service = InMemorySessionService()
workflow_runner = Runner(
    agent=greeter_agent,
    app_name=APP_NAME,
    session_service=workflow_session_service,
)

print(
    json.dumps(
        {
            "root_agent": greeter_agent.name,
            "root_sub_agents": [
                agent.name for agent in greeter_agent.sub_agents
            ],
            "workflow_type": type(answer_team).__name__,
            "workflow_order": [
                agent.name for agent in answer_team.sub_agents
            ],
            "state_handoffs": {
                "search": search_agent.output_key,
                "critique": critique_agent.output_key,
                "refine": refine_agent.output_key,
            },
            "search_tool": "google_search",
            "critique_tool": "google_search",
            "runtime_date_utc": CURRENT_DATE_UTC,
            "model": MODEL,
        },
        indent=2,
    )
)


/var/tmp/ipykernel_72598/505311504.py:124: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_team = SequentialAgent(
App "task4_agent_workflow" can transfer between agents but has no context_cache_config. Every transfer swaps the system instruction and the tool set, so the request prefix changes and the whole prompt is re-sent uncached after each transfer. Set context_cache_config on the app to give each agent its own cache.


{
  "root_agent": "greeter",
  "root_sub_agents": [
    "answer_team"
  ],
  "workflow_type": "SequentialAgent",
  "workflow_order": [
    "search_agent",
    "critique_agent",
    "refine_agent"
  ],
  "state_handoffs": {
    "search": "initial_answer",
    "critique": "critique",
    "refine": "refined_answer"
  },
  "search_tool": "google_search",
  "critique_tool": "google_search",
  "runtime_date_utc": "2026-08-20",
  "model": "gemini-2.5-flash"
}


## 3. Check the workflow setup

These checks confirm the agents, parent and child structure, stage order,
Google Search tools, and state handoffs. They do not spend model quota.


In [4]:
EXPECTED_STAGE_ORDER = [
    "search_agent",
    "critique_agent",
    "refine_agent",
]

architecture_evidence = {
    "source_notebook": TASK4_SOURCE_NOTEBOOK,
    "root_name": greeter_agent.name,
    "root_sub_agents": [
        agent.name for agent in greeter_agent.sub_agents
    ],
    "workflow_class": type(answer_team).__name__,
    "workflow_order": [
        agent.name for agent in answer_team.sub_agents
    ],
    "search_uses_builtin_google_search": (
        len(search_agent.tools) == 1
        and search_agent.tools[0] is google_search
    ),
    "critique_uses_builtin_google_search": (
        len(critique_agent.tools) == 1
        and critique_agent.tools[0] is google_search
    ),
    "state_output_keys": [
        search_agent.output_key,
        critique_agent.output_key,
        refine_agent.output_key,
    ],
}

assert TASK4_SOURCE_NOTEBOOK == "03_multi_agent_system.ipynb"
assert architecture_evidence["root_name"] == "greeter"
assert architecture_evidence["root_sub_agents"] == ["answer_team"]
assert architecture_evidence["workflow_class"] == "SequentialAgent"
assert architecture_evidence["workflow_order"] == EXPECTED_STAGE_ORDER
assert architecture_evidence["search_uses_builtin_google_search"] is True
assert architecture_evidence["critique_uses_builtin_google_search"] is True
assert architecture_evidence["state_output_keys"] == [
    "initial_answer",
    "critique",
    "refined_answer",
]
assert answer_team.parent_agent is greeter_agent
assert all(
    agent.parent_agent is answer_team
    for agent in answer_team.sub_agents
)
print(json.dumps(architecture_evidence, indent=2))


{
  "source_notebook": "03_multi_agent_system.ipynb",
  "root_name": "greeter",
  "root_sub_agents": [
    "answer_team"
  ],
  "workflow_class": "SequentialAgent",
  "workflow_order": [
    "search_agent",
    "critique_agent",
    "refine_agent"
  ],
  "search_uses_builtin_google_search": true,
  "critique_uses_builtin_google_search": true,
  "state_output_keys": [
    "initial_answer",
    "critique",
    "refined_answer"
  ]
}


## 4. Save a useful workflow log

ADK events show each step that ran. The short records keep the agent name,
branch, handoff, tool names, Google Search grounding status, changed state
keys, and final-answer marker. They leave out credentials, full provider
responses, and authenticated URLs.


In [5]:
def bounded_text(value: str | None, limit: int = 2400) -> str:
    """Normalize and truncate text for readable grading output."""
    if not value:
        return ""
    normalized = " ".join(value.split())
    return normalized[:limit] + (
        "..." if len(normalized) > limit else ""
    )


def validate_workflow_prompt(prompt: str) -> dict[str, Any]:
    """Reject blank or unreasonably large prompts before any model call."""
    normalized = " ".join(prompt.split())
    if not normalized:
        return {
            "ok": False,
            "error": "Please provide a nonempty question.",
        }
    if len(normalized) > 1200:
        return {
            "ok": False,
            "error": "Question exceeds the 1,200-character limit.",
        }
    return {"ok": True, "prompt": normalized}


def workflow_event_record(event: Any) -> dict[str, Any]:
    """Convert one ADK event into a compact, credential-free record."""
    calls = [call.name for call in event.get_function_calls()]
    state_delta = {}
    transfer_target = None
    if event.actions:
        state_delta = dict(event.actions.state_delta or {})
        transfer_target = event.actions.transfer_to_agent

    text = ""
    if event.content and event.content.parts:
        text = "".join(
            part.text or ""
            for part in event.content.parts
            if part.text
        )

    return {
        "author": event.author,
        "branch": event.branch,
        "transfer_to_agent": transfer_target,
        "function_calls": calls,
        "google_search_grounding": bool(
            getattr(event, "grounding_metadata", None)
        ),
        "state_delta_keys": sorted(state_delta),
        "is_final_response": event.is_final_response(),
        "text_preview": bounded_text(text, limit=320),
    }


async def run_workflow_case(
    prompt: str,
    *,
    label: str,
) -> dict[str, Any]:
    """Run one fresh-session workflow and return state plus event proof."""
    validation = validate_workflow_prompt(prompt)
    if not validation["ok"]:
        return {
            "label": label,
            "accepted": False,
            "error": validation["error"],
            "model_called": False,
        }

    session_id = f"task4-{label}-{uuid.uuid4().hex[:12]}"
    await workflow_session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
        state={"current_date_utc": CURRENT_DATE_UTC},
    )
    STAGE_START_EVENTS.clear()
    message = types.Content(
        role="user",
        parts=[types.Part.from_text(text=validation["prompt"])],
    )

    records: list[dict[str, Any]] = []
    final_answer = ""
    async with asyncio.timeout(150):
        async for event in workflow_runner.run_async(
            user_id=USER_ID,
            session_id=session_id,
            new_message=message,
        ):
            record = workflow_event_record(event)
            records.append(record)
            if record["is_final_response"] and event.content:
                candidate = " ".join(
                    part.text or ""
                    for part in event.content.parts
                    if part.text
                ).strip()
                if candidate:
                    final_answer = candidate

    completed_session = await workflow_session_service.get_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=session_id,
    )
    assert completed_session is not None
    state = completed_session.state
    stage_outputs = {
        key: str(state.get(key, "")).strip()
        for key in (
            "initial_answer",
            "critique",
            "refined_answer",
        )
    }

    event_authors = list(
        dict.fromkeys(record["author"] for record in records)
    )
    observed_stage_order = [
        author
        for author in event_authors
        if author in EXPECTED_STAGE_ORDER
    ]
    transfer_targets = [
        record["transfer_to_agent"]
        for record in records
        if record["transfer_to_agent"]
    ]

    return {
        "label": label,
        "accepted": True,
        "session_id": session_id,
        "started_agents": [
            item["agent"] for item in STAGE_START_EVENTS
        ],
        "event_authors": event_authors,
        "observed_stage_order": observed_stage_order,
        "transfer_targets": transfer_targets,
        "google_search_grounded": any(
            record["google_search_grounding"] for record in records
        ),
        "grounded_agents": list(
            dict.fromkeys(
                record["author"]
                for record in records
                if record["google_search_grounding"]
            )
        ),
        "initial_answer": bounded_text(
            stage_outputs["initial_answer"]
        ),
        "critique": bounded_text(stage_outputs["critique"]),
        "refinement": bounded_text(
            stage_outputs["refined_answer"]
        ),
        "final_answer": bounded_text(final_answer),
        "events": records,
    }


## 5. Test the full workflow

Each valid test starts with the Greeter in a fresh session. One case contains a
wrong premise, so the workflow must correct it. The checks require Search,
Critique, and Refine to run in order. They also require live Google Search
grounding, three saved state values, and a final answer from Refine. A correct
draft may stay mostly the same.


In [6]:
LIVE_CASES = [
    {
        "label": "current_official_update",
        "prompt": (
            "According to the latest official USGS Cascades Volcano "
            "Observatory update, what are Mount Rainier's current "
            "Volcano Alert Level and Aviation Color Code, and what do "
            "they mean? Include the update date if one is stated."
        ),
    },
    {
        "label": "misleading_premise_correction",
        "prompt": (
            "The World Health Organization declared COVID-19 a pandemic "
            "on March 11, 2019. Verify that date, correct it if needed, "
            "and identify the organization that made the announcement."
        ),
    },
    {
        "label": "compact_boundary_answer",
        "prompt": (
            "In one compact paragraph, when did the International "
            "Astronomical Union classify Pluto as a dwarf planet, and "
            "what resolution defined the category?"
        ),
    },
]

live_results: list[dict[str, Any]] = []
for case in LIVE_CASES:
    result = await run_workflow_case(
        case["prompt"],
        label=case["label"],
    )
    assert result["accepted"] is True, result
    assert result["observed_stage_order"] == EXPECTED_STAGE_ORDER, result
    assert set(["greeter", *EXPECTED_STAGE_ORDER]) <= set(
        result["started_agents"]
    ), result
    assert "answer_team" in result["transfer_targets"], result
    assert result["google_search_grounded"] is True, result
    assert {"search_agent", "critique_agent"} <= set(
        result["grounded_agents"]
    ), result
    assert result["initial_answer"], result
    assert result["critique"], result
    assert result["refinement"], result
    assert result["final_answer"], result
    assert result["final_answer"] == result["refinement"], result
    assert "[cite:" not in result["final_answer"].lower(), result
    live_results.append(result)

assert len({item["session_id"] for item in live_results}) == len(
    live_results
)
correction_case = next(
    item
    for item in live_results
    if item["label"] == "misleading_premise_correction"
)
assert "2020" in correction_case["final_answer"], correction_case
print(json.dumps(live_results, indent=2))


/opt/micromamba/lib/python3.12/site-packages/google/adk/tools/transfer_to_agent_tool.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  function_decl = super()._get_declaration()
Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


[
  {
    "label": "current_official_update",
    "accepted": true,
    "session_id": "task4-current_official_update-1292009086d6",
    "started_agents": [
      "greeter",
      "search_agent",
      "critique_agent",
      "refine_agent"
    ],
    "event_authors": [
      "greeter",
      "search_agent",
      "critique_agent",
      "refine_agent"
    ],
    "observed_stage_order": [
      "search_agent",
      "critique_agent",
      "refine_agent"
    ],
    "transfer_targets": [
      "answer_team"
    ],
    "google_search_grounded": true,
    "grounded_agents": [
      "search_agent",
      "critique_agent"
    ],
    "initial_answer": "According to the latest official USGS Cascades Volcano Observatory (CVO) update, Mount Rainier's current Volcano Alert Level is \"NORMAL\" and its Aviation Color Code is \"GREEN.\" This update was issued on Friday, August 14, 2026, at 11:06 AM PDT (18:06 UTC) as part of the Cascade Range Weekly Update. Here's what these designations mean: * **V

## 6. Reject bad input early

The notebook rejects blank or oversized input before it creates a session or
calls Gemini. This saves quota and keeps malformed requests out of the
workflow.


In [7]:
rejected_cases = [
    await run_workflow_case("   ", label="blank_question"),
    await run_workflow_case("x" * 1201, label="oversized_question"),
]
assert all(item["accepted"] is False for item in rejected_cases)
assert all(item["model_called"] is False for item in rejected_cases)
assert all(item["error"] for item in rejected_cases)
print(json.dumps(rejected_cases, indent=2))


[
  {
    "label": "blank_question",
    "accepted": false,
    "error": "Please provide a nonempty question.",
    "model_called": false
  },
  {
    "label": "oversized_question",
    "accepted": false,
    "error": "Question exceeds the 1,200-character limit.",
    "model_called": false
  }
]


## 7. Grading evidence

The final assertions connect each requirement to saved information about the
agents, events, state, grounding, answers, and rejected input.


In [8]:
correction_result = next(
    item
    for item in live_results
    if item["label"] == "misleading_premise_correction"
)

grading_evidence = {
    "copied_from_previous_notebook": (
        TASK4_SOURCE_NOTEBOOK == "03_multi_agent_system.ipynb"
    ),
    "greeter_agent_created_as_root": (
        greeter_agent.name == "greeter"
        and workflow_runner.agent is greeter_agent
    ),
    "search_agent_created": search_agent.name == "search_agent",
    "critique_agent_created": critique_agent.name == "critique_agent",
    "refine_agent_created": refine_agent.name == "refine_agent",
    "sequential_answer_team_created": (
        type(answer_team).__name__ == "SequentialAgent"
        and architecture_evidence["workflow_order"]
        == EXPECTED_STAGE_ORDER
    ),
    "search_uses_google_search": architecture_evidence[
        "search_uses_builtin_google_search"
    ],
    "critique_rechecks_with_google_search": architecture_evidence[
        "critique_uses_builtin_google_search"
    ],
    "initial_answer_saved_to_state": all(
        result["initial_answer"] for result in live_results
    ),
    "critique_saved_to_state": all(
        result["critique"] for result in live_results
    ),
    "refinement_saved_to_state": all(
        result["refinement"] for result in live_results
    ),
    "stages_ran_in_required_order": all(
        result["observed_stage_order"] == EXPECTED_STAGE_ORDER
        for result in live_results
    ),
    "greeter_delegated_to_workflow": all(
        "answer_team" in result["transfer_targets"]
        for result in live_results
    ),
    "events_prove_every_sub_agent": all(
        set(EXPECTED_STAGE_ORDER) <= set(result["event_authors"])
        for result in live_results
    ),
    "live_google_search_grounding_saved": all(
        {"search_agent", "critique_agent"}
        <= set(result["grounded_agents"])
        for result in live_results
    ),
    "initial_critique_refinement_final_visible": all(
        result["initial_answer"]
        and result["critique"]
        and result["refinement"]
        and result["final_answer"]
        for result in live_results
    ),
    "final_answer_is_refined_answer": all(
        result["final_answer"] == result["refinement"]
        for result in live_results
    ),
    "no_invented_citation_placeholders": all(
        "[cite:" not in result["final_answer"].lower()
        for result in live_results
    ),
    "misleading_premise_corrected": (
        "2020" in correction_result["final_answer"]
    ),
    "at_least_one_answer_changed_after_critique": any(
        result["initial_answer"] != result["refinement"]
        for result in live_results
    ),
    "fresh_session_for_every_live_case": len(
        {result["session_id"] for result in live_results}
    )
    == len(live_results),
    "invalid_input_rejected_before_model": all(
        not item["accepted"] and not item["model_called"]
        for item in rejected_cases
    ),
    "all_live_responses_nonempty": all(
        result["final_answer"] for result in live_results
    ),
}

assert all(grading_evidence.values()), grading_evidence
print(json.dumps(grading_evidence, indent=2))
print("TASK 4 COMPLETE: all agent-workflow grading checks passed.")


{
  "copied_from_previous_notebook": true,
  "greeter_agent_created_as_root": true,
  "search_agent_created": true,
  "critique_agent_created": true,
  "refine_agent_created": true,
  "sequential_answer_team_created": true,
  "search_uses_google_search": true,
  "critique_rechecks_with_google_search": true,
  "initial_answer_saved_to_state": true,
  "critique_saved_to_state": true,
  "refinement_saved_to_state": true,
  "stages_ran_in_required_order": true,
  "greeter_delegated_to_workflow": true,
  "events_prove_every_sub_agent": true,
  "live_google_search_grounding_saved": true,
  "initial_critique_refinement_final_visible": true,
  "final_answer_is_refined_answer": true,
  "no_invented_citation_placeholders": true,
  "misleading_premise_corrected": true,
  "at_least_one_answer_changed_after_critique": true,
  "fresh_session_for_every_live_case": true,
  "invalid_input_rejected_before_model": true,
  "all_live_responses_nonempty": true
}
TASK 4 COMPLETE: all agent-workflow grading c

## References

- [Google ADK workflow patterns](https://adk.dev/workflows/patterns/)
- [Google ADK sequential workflow agents](https://adk.dev/agents/workflow-agents/sequential-agents/)
- [Google ADK session state and `output_key`](https://adk.dev/sessions/state/)
- [Google ADK events](https://adk.dev/events/)
- [Google Search tool for ADK](https://adk.dev/tools/gemini-api/google-search/)
